<a href="https://colab.research.google.com/github/bcbutler2212/Education-Policy-Research-Chatbot/blob/Sofia/working_llama_3_2_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-1B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-1B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import os
from google.colab import userdata

# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
# load files temporarily
from google.colab import files
uploaded = files.upload()

Saving pdfs_for_chatbot.zip to pdfs_for_chatbot.zip


In [ ]:
# unzip
!unzip /content/pdfs_for_chatbot.zip

Archive:  /content/pdfs_for_chatbot.zip
   creating: pdfs_for_chatbot/
  inflating: pdfs_for_chatbot/AcademicStandards_Bleiberg_Formatted.pdf  
  inflating: pdfs_for_chatbot/AssistantPrincipals_Goldring_Formatted.pdf  
  inflating: pdfs_for_chatbot/BartanenRogers_AdministratorMobility_Formatted.pdf  
  inflating: pdfs_for_chatbot/BroaderECEBenefits_Gibbs_Formatted.pdf  
  inflating: pdfs_for_chatbot/Charters_Harris_Formatted.pdf  
  inflating: pdfs_for_chatbot/Choice_Carlson_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeAccess_Meyer_etal_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeAdmissions_Klasik_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegePricing_CookTurner_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeScholarships_Gurantz_Formatted.pdf  
  inflating: pdfs_for_chatbot/COVIDHigherEducation_ActonCook_Formatted.pdf  
  inflating: pdfs_for_chatbot/Covid_Zamarro_Formatted.pdf  
  inflating: pdfs_for_chatbot/Curriculum_Steiner_Formatted.pdf  
  inflating: p

In [ ]:
pip install langchain[all] langchain-community chromadb sentence-transformers transformers pypdf


In [ ]:
!pip install langchain-classic

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.vectorstores import Chroma
from langchain_classic.embeddings import SentenceTransformerEmbeddings
from langchain_classic.llms import HuggingFacePipeline
from langchain_classic.document_loaders import PyPDFLoader
from transformers import LlamaTokenizer, LlamaForCausalLM, pipeline



In [ ]:
from langchain_classic.chains import RetrievalQA

In [ ]:

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
import os
os.listdir("/content")

['.config', 'pdfs_for_chatbot', 'pdfs_for_chatbot.zip', 'sample_data']

In [ ]:
# Load all pdfs

pdf_folder = "/content/pdfs_for_chatbot"

docs = []
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        loaded_docs = loader.load()

        # Add source metadata for citations
        for i, doc in enumerate(loaded_docs):
            doc.metadata = {
                "source": f"{file} page {i+1}"
            }
        docs.extend(loader.load())

print(f"Loaded {len(docs)} PDF pages")

Loaded 815 PDF pages


In [ ]:

# Split long text into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)

print(f"Created {len(split_docs)} text chunks")

Created 3765 text chunks


In [ ]:
# wrap
#from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
#embedding model
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Chroma

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Create a persistent Chroma DB
vectordb = Chroma.from_documents(split_docs, embedding=embedding_model, persist_directory="chroma_db")


In [ ]:
vectordb.persist()

In [ ]:
# If chunks print then working

results = vectordb.similarity_search("What is this document about?", k=2)
for i, r in enumerate(results, 1):
    print(f"\nResult {i}:\n{r.page_content[:500]}...")


Result 1:
bring attention to an issue, expand a person’s underȉanding of an area of policy or praǻice, or 
Ǻange the way in whiǺ someone looks at a problem.  
 
While ȉudies provide empirical support for conceptual use, measuring this type of use is 
more Ǻallenging. Moȉ researǺ to date involves self-reports through interviews or surveys. 
Furthermore, it does not examine the ways in whiǺ conceptual use may precede or be 
embedded in inȉrumental or symbolic use of researǺ evidence, for example, when 
rese...

Result 2:
bring attention to an issue, expand a person’s underȉanding of an area of policy or praǻice, or 
Ǻange the way in whiǺ someone looks at a problem.  
 
While ȉudies provide empirical support for conceptual use, measuring this type of use is 
more Ǻallenging. Moȉ researǺ to date involves self-reports through interviews or surveys. 
Furthermore, it does not examine the ways in whiǺ conceptual use may precede or be 
embedded in inȉrumental or symbolic use of researǺ evidenc

In [ ]:
# make retriever --> return 3 chunks? change this?
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain_classic.chains import RetrievalQA

# connect llama with chroma
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [ ]:
# test it!
# result = qa_chain.invoke({"query": "Summarize the key findings from the PDFs"})
# print(result["result"])

In [ ]:
question = "What is education policy?"
result = qa_chain.invoke({"query": question})
print(result["result"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

School Funding. Policy Brief. Public Policy Institute of California. 
26 Costrell, Robert M., and Josh B. McGee. 2022. Recent Research on Teacher Pension Funding, Benefits, and 
Policy Debates. In Recent Advancements in Education Finance and Policy, Downes, T.A., K.M. Killeen (Eds.), 
Information Age Publishing.

School Funding. Policy Brief. Public Policy Institute of California. 
26 Costrell, Robert M., and Josh B. McGee. 2022. Recent Research on Teacher Pension Funding, Benefits, and 
Policy Debates. In Recent Advancements in Education Finance and Policy, Downes, T.A., K.M. Killeen (Eds.), 
Information Age Publishing.

Evidence 
Key finding #1: Good education policy, as well as useful education policy research, requires 
careful attention to and trade-offs among the many values at stake. 
This handbook provides extensive 

In [ ]:
#this prints the sources
if "source_documents" in result:
    sources = set()
    for doc in result["source_documents"]:
        src = doc.metadata.get("source", "unknown")
        sources.add(src)

    print("\nSources:")
    for src in sources:
        print("-", src)


Sources:
- /content/pdfs_for_chatbot/SchoolFunding_LafortuneMcGee_Formatted.pdf
- /content/pdfs_for_chatbot/EducationalGoods_Loeb_Formatted.pdf
